# Kalshi DF Builder

In [2]:
import requests
import pandas as pd

BASE = "https://api.elections.kalshi.com/trade-api/v2"

params = {
    "limit": 100,
    "with_nested_markets": "true",
}

r = requests.get(f"{BASE}/events", params=params, timeout=15)
r.raise_for_status()
events = r.json().get("events", [])

rows = []

for ev in events:
    title = ev.get("title")

    binary_markets = [
        m for m in ev.get("markets", [])
        if m.get("market_type") == "binary"
    ]

    # exclude non-naturally-binary events
    if len(binary_markets) != 1:
        continue

    m = binary_markets[0]

    yes = m.get("last_price_dollars") or m.get("last_price")
    no  = m.get("no_ask_dollars") or m.get("no_ask")

    if float(yes or 0) == 0.0 and float(no or 1) == 1.0:
        continue

    rows.append({
        "event_title": title,
        "market_title": m.get("title"),
        "yes_price": yes,
        "no_price": no,
        "volume": m.get("volume"),
    })


df = pd.DataFrame(rows)
print(df.head(20))

                                          event_title  \
0          Will Elon Musk visit Mars in his lifetime?   
1   Will the world pass 2 degrees Celsius over pre...   
2   Will a human land on Mars before California st...   
3              Will a supervolcano erupt before 2050?   
4              Will humans colonize Mars before 2050?   
5   Will Zohran Mamdani become President of the Un...   
6   Will Nick Fuentes become President of the Unit...   
7   Will any U.S. state experience a population de...   
8   Will there be an at least 8.0 magnitude earthq...   
9   Will a humanoid robot walk on Mars before a hu...   
10  Will Johnny Depp be cast in the next Pirates o...   
11  Will Andrew Tate's party win a seat in the nex...   
12       Initial jobless claims from Aug 22-28, 2021?   
13  Will the FDA approve a cure for Type 1 diabete...   
14                   EU meets its 2030 climate goals?   
15                India meets its 2030 climate goals?   
16  Will Tom Cruise be cast in 

# Polymarket DF Builder

In [53]:
import requests, json, time
import pandas as pd

GAMMA = "https://gamma-api.polymarket.com/events"
CLOB = "https://clob.polymarket.com/book"   # used only if VERIFY_BOOK True

TARGET = 10
PAGE_SIZE = 100
VOLUME_MIN = 100.0
VERIFY_BOOK = False   # set True to require non-empty order books (slower)

def parse_list(v):
    if isinstance(v, str):
        try: return json.loads(v)
        except: return []
    return v or []

def has_book_liquidity(token_id):
    try:
        r = requests.get(CLOB, params={"token_id": token_id}, timeout=5)
        r.raise_for_status()
        b = r.json()
        return bool(b.get("bids")) or bool(b.get("asks"))
    except Exception:
        return False

rows = {}
offset = 0

while len(rows) < TARGET:
    params = {
        "limit": PAGE_SIZE,
        "offset": offset,
        "closed": "false",
        "active": "true",
        "with_nested_markets": "true",
        "order": "id",
        "ascending": "false",
    }
    r = requests.get(GAMMA, params=params, timeout=12)
    r.raise_for_status()
    payload = r.json()
    evs = payload if isinstance(payload, list) else payload.get("events", []) or payload.get("data", [])
    if not evs:
        break

    for ev in evs:
        for m in ev.get("markets", []) or ev.get("nested_markets", []) or []:
            # require two outcomes (binary) and yes/no labels
            outcomes = parse_list(m.get("outcomes") or m.get("outcome_labels") or [])
            if not (isinstance(outcomes, list) and len(outcomes) == 2):
                continue
            names = [str(x).strip().lower() for x in outcomes]
            if not (("yes" in names[0] or "true" in names[0] or "y" in names[0]) and
                    ("no" in names[1] or "false" in names[1] or "n" in names[1]) or
                    (("yes" in names[1] or "true" in names[1]) and ("no" in names[0] or "false" in names[0]))):
                continue

            # parse outcomePrices (Gamma often stringifies arrays)
            prices = parse_list(m.get("outcomePrices") or m.get("prices") or [])
            if not (isinstance(prices, list) and len(prices) == 2):
                # try fallback to token.price fields
                tokens = m.get("tokens") or []
                if isinstance(tokens, str):
                    try: tokens = json.loads(tokens)
                    except: tokens = []
                if isinstance(tokens, list) and len(tokens) >= 2:
                    p0 = tokens[0].get("price") if isinstance(tokens[0], dict) else None
                    p1 = tokens[1].get("price") if isinstance(tokens[1], dict) else None
                    try:
                        prices = [float(p0), float(p1)]
                    except Exception:
                        continue
                else:
                    continue

            try:
                yes, no = float(prices[0]), float(prices[1])
            except Exception:
                continue

            # skip placeholder/resolved prices
            if yes == 0.0 and no == 0.0:
                continue

            # volume filter
            vol = m.get("volumeNum") or m.get("volume") or ev.get("volumeNum") or 0
            try: vol = float(vol)
            except: vol = 0.0
            if vol < VOLUME_MIN:
                continue

            # optional: verify order book liquidity (slower but authoritative)
            if VERIFY_BOOK:
                # find token ids for yes/no in tokens array
                tokens = m.get("tokens") or []
                if isinstance(tokens, str):
                    try: tokens = json.loads(tokens)
                    except: tokens = []
                tid_yes = tid_no = None
                for t in tokens:
                    if not isinstance(t, dict): continue
                    o = (t.get("outcome") or "").lower()
                    if "yes" in o or "true" in o:
                        tid_yes = t.get("token_id") or t.get("tokenId") or t.get("id")
                    if "no" in o or "false" in o:
                        tid_no  = t.get("token_id") or t.get("tokenId") or t.get("id")
                if not ( (tid_yes and has_book_liquidity(tid_yes)) or (tid_no and has_book_liquidity(tid_no)) ):
                    continue

            rows[m.get("id") or m.get("market_id") or (ev.get("id") + ":" + str(m.get("id")))] = {
                "event": ev.get("title") or ev.get("question") or ev.get("name"),
                "market": m.get("title") or m.get("name") or m.get("question"),
                "yes": round(yes, 6),
                "no": round(no, 6),
                "volume": vol,
                "source": "gamma",
            }
            if len(rows) >= TARGET:
                break
        if len(rows) >= TARGET:
            break

    print("offset", offset, "collected", len(rows), "/", TARGET)
    offset += PAGE_SIZE
    time.sleep(0.2)

df = pd.DataFrame(list(rows.values()))
print(df)






offset 0 collected 1 / 10
offset 100 collected 10 / 10
                                         event  \
0  Will Trump meet with Machado by January 16?   
1                   2026 NFL Wild Card Parlays   
2                   2026 NFL Wild Card Parlays   
3                   2026 NFL Wild Card Parlays   
4                   2026 NFL Wild Card Parlays   
5                   2026 NFL Wild Card Parlays   
6                   2026 NFL Wild Card Parlays   
7                   2026 NFL Wild Card Parlays   
8                   2026 NFL Wild Card Parlays   
9                   2026 NFL Wild Card Parlays   

                                              market    yes     no  \
0        Will Trump meet with Machado by January 16?  0.855  0.145   
1  Will LAR/GB/BUF/SF/LAC/HOU win their 2026 NFL ...  0.040  0.960   
2  Will LAR/GB/BUF/SF/LAC/PIT win their 2026 NFL ...  0.040  0.960   
3  Will LAR/GB/BUF/SF/NE/PIT win their 2026 NFL W...  0.040  0.960   
4  Will LAR/GB/BUF/PHI/LAC/PIT win their 202